# 04 — EventWarpNet Training (Flow-Based)
TimeLens-style flow estimation + warping + refinement.

**Why:** DualEncoderUNet (direct synthesis) plateaus at 31 dB with residual std=0.05 — it barely deviates from the mean frame. EventWarpNet explicitly estimates flow from events, warps frames, then refines. Warping preserves sharpness; the refinement network only handles occlusions.

**Architecture:**
- FlowNet: estimates bidirectional flow from events + RGB (11ch → 4ch)
- ContextEncoder: extracts features from f0, f1
- RefineNet: predicts alpha mask + residual from warped frames + events
- Output: `alpha * warped_f0 + (1-alpha) * warped_f1 + residual`

In [1]:
# ── Cell 1: Setup & Mount ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/493Project')

%pip install -q torchmetrics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.1 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os
import glob
import random
import torch
from torch.amp import GradScaler
from torch.utils.data import DataLoader

from src.data import VimeoTripletDataset
from src.train import (
    build_model,
    build_criterion,
    build_optimizer,
    build_scheduler,
    load_checkpoint,
    overfit_one_batch,
    run_training,
)
from src.losses import CharbonnierLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [3]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
cfg = dict(
    # Paths
    vimeo_root     = '/content/drive/MyDrive/493Project/data/vimeo',
    local_root     = '/content/local_vimeo',
    checkpoint_dir = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v7_warp',

    # Model — EventWarpNet (flow-based)
    model_type     = 'warp',

    # Data
    patch_size     = 256,
    batch_size     = 8,        # smaller batch — EventWarpNet uses more VRAM (warping + context)
    num_workers    = 4,

    # Optimizer
    lr             = 2e-4,
    lr_min         = 1e-6,
    weight_decay   = 0.0,

    # Scheduler
    warmup_epochs  = 0,

    # Loss — Charb + flow smoothness (flow-based model benefits from smoothness)
    loss_type      = 'combined',
    lambda_char            = 1.0,
    lambda_perceptual      = 0.0,    # start Charb-only, add perceptual later
    lambda_ssim            = 0.0,
    lambda_smooth          = 0.01,   # flow smoothness — important for EventWarpNet
    lambda_event_weighted  = 0.0,

    # Training
    num_epochs     = 40,
    max_norm       = 1.0,
)

os.makedirs(cfg['checkpoint_dir'], exist_ok=True)
print(f'Model: {cfg["model_type"]}')
print(f'Loss: Charb({cfg["lambda_char"]}) + Smooth({cfg["lambda_smooth"]})')
print(f'LR: {cfg["lr"]} | Batch: {cfg["batch_size"]} | Epochs: {cfg["num_epochs"]}')

Model: warp
Loss: Charb(1.0) + Smooth(0.01)
LR: 0.0002 | Batch: 8 | Epochs: 40


In [4]:
# ── Cell 4: Load data — extract tar to local SSD ────────────────────────────
import subprocess

vimeo_root = cfg['vimeo_root']
local_root = cfg['local_root']
os.makedirs(local_root, exist_ok=True)

tar_on_drive = os.path.join(vimeo_root, 'processed.tar')
marker = os.path.join(local_root, '.copy_done')

if not os.path.exists(marker):
    assert os.path.exists(tar_on_drive), \
        f'processed.tar not found on Drive — run 00b_tar_processed.ipynb first'

    subprocess.run(['apt-get', 'install', '-qq', '-y', 'pv'], capture_output=True)
    total_bytes = os.path.getsize(tar_on_drive)
    print(f'Extracting processed.tar ({total_bytes / 1e9:.1f} GB) to local SSD...')
    subprocess.run(
        f'pv -f -s {total_bytes} "{tar_on_drive}" | tar xf - -C "{local_root}"',
        shell=True, check=True,
    )
    open(marker, 'w').close()
    print('Done.')
else:
    print('Local data already exists, skipping extraction.')

def load_split(split_file):
    path = os.path.join(vimeo_root, split_file)
    with open(path) as f:
        entries = [line.strip() for line in f if line.strip()]
    dirs = []
    for rel in entries:
        d = os.path.join(local_root, rel)
        if os.path.isdir(d):
            dirs.append((rel, d))
    return dirs

train_entries = load_split('tri_trainlist.txt')
val_entries   = load_split('tri_vallist.txt')

if len(val_entries) == 0 and len(train_entries) > 0:
    print(f'No processed val triplets found — splitting 90/10 from {len(train_entries)} train entries')
    rng = random.Random(42)
    all_entries = list(train_entries)
    rng.shuffle(all_entries)
    n_val = max(1, int(len(all_entries) * 0.10))
    val_entries   = all_entries[:n_val]
    train_entries = all_entries[n_val:]

train_dirs = [d for _, d in train_entries]
val_dirs   = [d for _, d in val_entries]
print(f'Train: {len(train_dirs)} | Val: {len(val_dirs)}')

AssertionError: processed.tar not found on Drive — run 00b_tar_processed.ipynb first

In [ ]:
# ── Cell 5: Create datasets and dataloaders ──────────────────────────────────
train_dataset = VimeoTripletDataset(train_dirs, patch_size=cfg['patch_size'], augment=True)
val_dataset   = VimeoTripletDataset(val_dirs,   patch_size=cfg['patch_size'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg['batch_size'], shuffle=False,
                          num_workers=cfg['num_workers'], pin_memory=True)

print(f'Train: {len(train_dataset)} clips, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} clips, {len(val_loader)} batches')

# Quick shape check
f0, f1, voxel, gt = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  f0:    {tuple(f0.shape)}')
print(f'  f1:    {tuple(f1.shape)}')
print(f'  voxel: {tuple(voxel.shape)}')
print(f'  gt:    {tuple(gt.shape)}')

In [ ]:
# ── Cell 6: Model, optimizer, scheduler, loss ────────────────────────────────
model     = build_model(cfg, device)
optimizer = build_optimizer(model, cfg)
scheduler = build_scheduler(optimizer, cfg)
scaler    = GradScaler('cuda', enabled=False)
criterion = build_criterion(cfg, device)

# Resume from checkpoint if one exists
start_epoch, best_psnr = load_checkpoint(
    cfg['checkpoint_dir'], model, optimizer, scheduler, device
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: EventWarpNet | Parameters: {n_params:,}')
print(f'LR: {cfg["lr"]} | AMP: {scaler.is_enabled()}')
print(f'Loss: Charb({cfg["lambda_char"]}) + Smooth({cfg["lambda_smooth"]})')

In [ ]:
# ── Cell 7: Sanity check — overfit one batch ─────────────────────────────────
# EventWarpNet should hit 35+ dB on one sample in 500 iters if architecture is correct
overfit_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                            num_workers=0, pin_memory=True)
sane_model     = build_model(cfg, device)
sane_optimizer = torch.optim.Adam(sane_model.parameters(), lr=2e-3)
sane_criterion = CharbonnierLoss()

final_loss = overfit_one_batch(
    sane_model, overfit_loader, sane_criterion, sane_optimizer, device, iters=500
)
print(f'Overfit final loss: {final_loss:.6f}  (target < 0.005)')
del sane_model, sane_optimizer, sane_criterion, overfit_loader

In [ ]:
# ── Cell 8: Full training loop ───────────────────────────────────────────────
training_log = run_training(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler, scaler,
    device, cfg,
    start_epoch=start_epoch,
    best_psnr=best_psnr,
)
print('Training complete.')

In [ ]:
# ── Cell 9: Qualitative results + flow visualization ─────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image
from src.train import _forward

best_model = build_model(cfg, device)
best_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
best_model.eval()

def load_vimeo_clip(clip_dir):
    f0  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
    f1  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
    gt  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
    evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)
    return f0, f1, gt, evt

def flow_to_rgb(flow):
    """Convert (2, H, W) flow to RGB visualization."""
    u, v = flow[0].numpy(), flow[1].numpy()
    mag = np.sqrt(u**2 + v**2)
    angle = np.arctan2(v, u)
    hsv = np.zeros((*mag.shape, 3), dtype=np.uint8)
    hsv[..., 0] = ((angle + np.pi) / (2 * np.pi) * 179).astype(np.uint8)
    hsv[..., 1] = 255
    max_mag = mag.max() + 1e-8
    hsv[..., 2] = (np.clip(mag / max_mag, 0, 1) * 255).astype(np.uint8)
    import cv2
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

rng = random.Random(42)
show_dirs = rng.sample(val_dirs, min(6, len(val_dirs)))

cols = ['Frame f0', 'Frame f1', 'Ground Truth', 'Prediction', 'Error (x5)', 'Flow t→0', 'Flow t→1']
fig, axes = plt.subplots(len(show_dirs), 7, figsize=(32, 4.5 * len(show_dirs)))
if len(show_dirs) == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(cols):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

with torch.no_grad():
    for row, clip_dir in enumerate(show_dirs):
        f0, f1, gt, evt = load_vimeo_clip(clip_dir)
        _, H, W = f0.shape
        pad_h = (16 - H % 16) % 16
        pad_w = (16 - W % 16) % 16
        f0_d = torch.nn.functional.pad(f0, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
        f1_d = torch.nn.functional.pad(f1, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
        evt_d = torch.nn.functional.pad(evt, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)

        outputs = _forward(best_model, f0_d, f1_d, evt_d)
        pred = outputs[0].squeeze(0).float().cpu().clamp(0, 1)[:, :H, :W]
        flow_t0 = outputs[1].squeeze(0).float().cpu()[:, :H, :W]
        flow_t1 = outputs[2].squeeze(0).float().cpu()[:, :H, :W]

        error = (gt - pred).abs() * 5
        mse = ((gt - pred) ** 2).mean().item()
        psnr = -10 * np.log10(mse + 1e-10)

        images = [f0, f1, gt, pred, error]
        for col, img in enumerate(images):
            axes[row, col].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])

        # Flow visualizations
        try:
            axes[row, 5].imshow(flow_to_rgb(flow_t0))
            axes[row, 6].imshow(flow_to_rgb(flow_t1))
        except ImportError:
            # cv2 not available — show magnitude instead
            axes[row, 5].imshow(flow_t0.norm(dim=0).numpy(), cmap='hot')
            axes[row, 6].imshow(flow_t1.norm(dim=0).numpy(), cmap='hot')
        axes[row, 5].set_xticks([]); axes[row, 5].set_yticks([])
        axes[row, 6].set_xticks([]); axes[row, 6].set_yticks([])

        axes[row, 3].set_xlabel(f'PSNR: {psnr:.2f} dB', fontsize=9, color='steelblue')
        axes[row, 5].set_xlabel(f'max={flow_t0.abs().max():.1f}px', fontsize=8)
        axes[row, 6].set_xlabel(f'max={flow_t1.abs().max():.1f}px', fontsize=8)

plt.suptitle('EventWarpNet — Predictions + Estimated Flow Fields', fontsize=15, y=1.01)
plt.tight_layout()
fig_path = os.path.join(cfg['checkpoint_dir'], 'qualitative_results_with_flow.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved to {fig_path}')

In [ ]:
# ── Cell 10: GIF comparison — GT vs DualEncoder vs EventWarpNet ──────────────
import numpy as np
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
from src.train import _forward, build_model
from IPython.display import display, HTML
import base64

# Load DualEncoderUNet (best Charb-only) for comparison
dual_cfg = {**cfg, 'model_type': 'dual'}
dual_model = build_model(dual_cfg, device)
dual_ckpt = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v6_dual/best_model.pth'
dual_model.load_state_dict(torch.load(dual_ckpt, map_location=device))
dual_model.eval()

# EventWarpNet best
warp_model = build_model(cfg, device)
warp_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
warp_model.eval()

rng = random.Random(99)
gif_dirs = rng.sample(val_dirs, min(8, len(val_dirs)))

def to_pil(tensor):
    return Image.fromarray((tensor.permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8))

for i, clip_dir in enumerate(gif_dirs):
    f0  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
    f1  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
    gt  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
    evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)

    _, H, W = f0.shape
    pad_h = (16 - H % 16) % 16
    pad_w = (16 - W % 16) % 16
    f0_d = torch.nn.functional.pad(f0, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
    f1_d = torch.nn.functional.pad(f1, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)
    evt_d = torch.nn.functional.pad(evt, (0, pad_w, 0, pad_h)).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_dual = _forward(dual_model, f0_d, f1_d, evt_d)[0].squeeze(0).float().cpu().clamp(0, 1)[:, :H, :W]
        pred_warp = _forward(warp_model, f0_d, f1_d, evt_d)[0].squeeze(0).float().cpu().clamp(0, 1)[:, :H, :W]

    gt_frames   = [to_pil(f0), to_pil(gt),        to_pil(f1)]
    dual_frames = [to_pil(f0), to_pil(pred_dual), to_pil(f1)]
    warp_frames = [to_pil(f0), to_pil(pred_warp), to_pil(f1)]

    w, h = gt_frames[0].size
    gap = 6
    label_h = 22
    combined_frames = []
    labels = ['Ground Truth', 'DualEncoder', 'EventWarpNet']
    for g, d, wp in zip(gt_frames, dual_frames, warp_frames):
        combined = Image.new('RGB', (w * 3 + gap * 2, h + label_h), (255, 255, 255))
        combined.paste(g, (0, label_h))
        combined.paste(d, (w + gap, label_h))
        combined.paste(wp, (w * 2 + gap * 2, label_h))
        draw = ImageDraw.Draw(combined)
        for col_idx, lbl in enumerate(labels):
            x_pos = col_idx * (w + gap) + w // 2 - len(lbl) * 3
            draw.text((x_pos, 3), lbl, fill=(0, 0, 0))
        combined_frames.append(combined)

    gif_path = os.path.join(cfg['checkpoint_dir'], f'comparison_3way_gif_{i}.gif')
    combined_frames[0].save(
        gif_path, save_all=True, append_images=combined_frames[1:],
        duration=400, loop=0,
    )

    with open(gif_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<p><b>Sample {i}</b></p>'
        f'<img src="data:image/gif;base64,{b64}" />'
    ))

print(f'\nSaved {len(gif_dirs)} GIFs to {cfg["checkpoint_dir"]}')